In [3]:
pip install -q -U google-genai

In [6]:
from google import genai
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY").strip()
client = genai.Client(api_key=key)   # keep it in a variable

names = [m.name for m in client.models.list()]
print("gemini-3.5-flash available:", "models/gemini-3.5-flash" in names)
print([n for n in names if "flash" in n])

gemini-3.5-flash available: True
['models/gemini-2.5-flash', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-omni-1.1-flash', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash', 'models/gemini-3.1-flash-tts-preview', 'models/gemini-3.8-flash-tts', 'models/gemini-3.8-flash-lite-tts', 'models/gemini-2.5-flash-native-audio-latest', 'models/gemini-2.5-flash-native-audio-preview-09-2025', 'models/gemini-2.5-flash-native-audio-preview-12-2025', 'models/gemini-3.1-flash-live-preview']


In [7]:
%%writefile agent.py
import json
import os
import time
from google import genai
from google.genai import types

MODEL = "gemini-3.5-flash"
MAX_RETRIES = 1
CALLS = 0


class APIError(Exception):
    """API/quota problem: stop immediately instead of wasting more calls."""


# ---------------- LLM wrapper (swap this one function for another provider) ----------------
def get_key() -> str:
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY").strip()
    except Exception:
        return os.environ["GEMINI_API_KEY"].strip()


def call_llm(prompt: str) -> str:
    global CALLS
    client = genai.Client(api_key=get_key())
    for attempt in range(3):
        time.sleep(2)
        CALLS += 1
        try:
            resp = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(response_mime_type="application/json"),
            )
            return resp.text
        except Exception as e:
            if "503" in str(e) and attempt < 2:  # temporary overload only
                time.sleep(15)
                continue
            raise APIError(str(e)[:300])


def parse_json(text: str) -> dict:
    text = text.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(text)


# ---------------- Prompts (all in one place) ----------------
PLANNER_PROMPT = """You are a planner. Given a word problem, output a short numbered plan (3-6 steps).
Do NOT solve the problem. Output ONLY JSON: {{"plan": ["step 1", "step 2", ...]}}

Example 1:
Question: Alice has 3 red apples and twice as many green apples as red. How many apples in total?
{{"plan": ["Red = 3", "Green = 2 x red", "Total = red + green", "Format answer"]}}

Example 2:
Question: A train leaves at 14:30 and arrives at 18:05. How long is the journey?
{{"plan": ["Convert both times to minutes", "Subtract (add 24h if arrival is next day)", "Convert back to hours and minutes", "Format answer"]}}

Example 3:
Question: A meeting needs 60 minutes. Free slots: 09:00-09:30, 09:45-10:30. Which fit?
{{"plan": ["Compute length of each slot", "Keep slots with length >= 60", "List them"]}}

{feedback}
Question: {question}
"""

EXECUTOR_PROMPT = """You are an executor. Follow the plan exactly and show intermediate calculations.
Output ONLY JSON: {{"steps": "<intermediate calculations>", "answer": "<short final answer>", "explanation": "<1-2 sentence user-friendly explanation>"}}

Example:
Question: Alice has 3 red apples and twice as many green apples as red. Total?
Plan: ["Red = 3", "Green = 2 x red", "Total = red + green"]
{{"steps": "red=3; green=2*3=6; total=3+6=9", "answer": "9 apples", "explanation": "Alice has 3 red and 6 green apples, so 9 in total."}}

Question: {question}
Plan: {plan}
"""

VERIFIER_PROMPT = """You are a verifier. First solve the question yourself, independently, WITHOUT looking at the proposed answer.
Then compare with the proposed solution. Decide same_answer by MEANING only: ignore wording, formatting, spelling of units and order of items.
Also check constraints (times valid, counts non-negative, totals consistent).
Output ONLY JSON:
{{"independent_answer": "<your answer>", "same_answer": true/false, "constraints_ok": true/false, "details": "<short reason>"}}

Example:
Question: Alice has 3 red apples and twice as many green. Total?
Proposed answer: 9 apples
{{"independent_answer": "9 apples", "same_answer": true, "constraints_ok": true, "details": "3 + 6 = 9, matches."}}

Example:
Question: Train leaves 14:30, arrives 18:05. Duration?
Proposed answer: 4 hours 35 minutes
{{"independent_answer": "3 hours 35 minutes", "same_answer": false, "constraints_ok": true, "details": "18:05 - 14:30 is 3h35m, not 4h35m."}}

Question: {question}
Proposed answer: {answer}
Proposed steps: {steps}
"""


# ---------------- The three phases ----------------
def planner(question: str, feedback: str = "") -> list:
    fb = f"A previous attempt failed verification because: {feedback}\nMake a better plan.\n" if feedback else ""
    return parse_json(call_llm(PLANNER_PROMPT.format(question=question, feedback=fb)))["plan"]


def executor(question: str, plan: list) -> dict:
    return parse_json(call_llm(EXECUTOR_PROMPT.format(question=question, plan=json.dumps(plan))))


def verifier(question: str, solution: dict) -> dict:
    return parse_json(call_llm(VERIFIER_PROMPT.format(
        question=question, answer=solution["answer"], steps=solution["steps"])))


# ---------------- Agent loop ----------------
def solve(question: str) -> dict:
    retries = 0
    feedback = ""
    while True:
        plan, solution = [], {}
        try:
            plan = planner(question, feedback)
            solution = executor(question, plan)
            v = verifier(question, solution)
        except (json.JSONDecodeError, KeyError, TypeError) as e:  # bad model output (APIError is NOT caught)
            v = {"same_answer": False, "constraints_ok": False,
                 "details": f"Malformed model output: {e}", "independent_answer": ""}

        same = str(v.get("same_answer")).lower() == "true"
        ok = str(v.get("constraints_ok")).lower() == "true"
        details = str(v.get("details"))
        checks = [
            {"check_name": "independent_resolve", "passed": same,
             "details": f"Verifier answer: {v.get('independent_answer')}. {details}"},
            {"check_name": "constraint_check", "passed": ok, "details": details},
        ]
        meta = {"plan": " | ".join(plan), "checks": checks, "retries": retries}

        if same and ok:
            return {"answer": solution.get("answer"), "status": "success",
                    "reasoning_visible_to_user": solution.get("explanation"), "metadata": meta}
        if retries >= MAX_RETRIES:
            return {"answer": "Could not produce a verified answer.", "status": "failed",
                    "reasoning_visible_to_user": f"Verification failed after {retries} retries: {details}",
                    "metadata": meta}
        retries += 1
        feedback = details


# ---------------- CLI (python agent.py) ----------------
if __name__ == "__main__":
    print("Reasoning agent. Type a question (or 'quit').")
    while True:
        q = input("\n> ").strip()
        if q.lower() in ("quit", "exit", ""):
            break
        print(json.dumps(solve(q), indent=2))

Writing agent.py


In [8]:
%%writefile tests.py
import json, os, sys
import agent

EASY = [
    ("If a train leaves at 14:30 and arrives at 18:05, how long is the journey?", ["3", "35"]),
    ("Alice has 3 red apples and twice as many green apples as red. How many apples in total?", ["9"]),
    ("A meeting needs 60 minutes. Free slots: 09:00-09:30, 09:45-10:30, 11:00-12:00. Which slots can fit the meeting?", ["11:00"]),
    ("A pen costs 12 rupees. How much do 7 pens cost?", ["84"]),
    ("Ravi is 25. His sister is 7 years younger. What is the sum of their ages?", ["43"]),
]

TRICKY = [
    ("A train leaves at 23:45 and arrives at 02:10 the next day. How long is the journey?", ["2", "25"]),
    ("Alice has 3 red apples and twice as many green apples as red. She gives away 2 green apples. How many does she have now?", ["7"]),
    ("A meeting needs 45 minutes. Free slots: 10:00-10:45, 11:00-11:40. Which slots can fit it?", ["10:00"]),
]

batch = sys.argv[1] if len(sys.argv) > 1 else "easy"
tests = EASY if batch == "easy" else TRICKY

results = json.load(open("run_logs.json")) if os.path.exists("run_logs.json") else []
done = {r["question"] for r in results}

for q, expected in tests:
    if q in done:
        continue
    try:
        out = agent.solve(q)
    except agent.APIError as e:
        print(f"\nSTOPPED (API/quota error, no more calls made): {e}")
        break
    correct = all(x in (out["answer"] or "") for x in expected)
    results.append({
        "level": batch, "question": q, "output": out,
        "verifier_passed": out["status"] == "success",
        "retried": out["metadata"]["retries"] > 0,
        "matches_expected": correct,
    })
    json.dump(results, open("run_logs.json", "w"), indent=2)  # saved after every question
    print(f"[{batch}] {q}\n  answer={out['answer']} | verifier_passed={out['status'] == 'success'} "
          f"| retries={out['metadata']['retries']} | correct={correct}\n")

print(f"Logged questions: {len(results)} | API calls made this run: {agent.CALLS}")

Writing tests.py


In [10]:
from google import genai
from google.colab import userdata
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY").strip())
print([m.name for m in client.models.list() if "flash" in m.name])

['models/gemini-2.5-flash', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-omni-1.1-flash', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash', 'models/gemini-3.1-flash-tts-preview', 'models/gemini-3.8-flash-tts', 'models/gemini-3.8-flash-lite-tts', 'models/gemini-2.5-flash-native-audio-latest', 'models/gemini-2.5-flash-native-audio-preview-09-2025', 'models/gemini-2.5-flash-native-audio-preview-12-2025', 'models/gemini-3.1-flash-live-preview']


In [1]:
%run tests.py easy

[easy] A meeting needs 60 minutes. Free slots: 09:00-09:30, 09:45-10:30, 11:00-12:00. Which slots can fit the meeting?
  answer=11:00-12:00 | verifier_passed=True | retries=0 | correct=True

[easy] A pen costs 12 rupees. How much do 7 pens cost?
  answer=84 rupees | verifier_passed=True | retries=0 | correct=True

[easy] Ravi is 25. His sister is 7 years younger. What is the sum of their ages?
  answer=43 | verifier_passed=True | retries=0 | correct=True

Logged questions: 5 | API calls made this run: 9


In [11]:
import re
NEW_MODEL = "gemini-3.1-flash-lite"

src = open("agent.py").read()
src = re.sub(r'MODEL = ".*?"', f'MODEL = "{NEW_MODEL}"', src)
open("agent.py", "w").write(src)
print([l for l in src.splitlines() if l.startswith("MODEL")])

['MODEL = "gemini-3.1-flash-lite"']


In [2]:
%run tests.py tricky

[tricky] A train leaves at 23:45 and arrives at 02:10 the next day. How long is the journey?
  answer=2 hours and 25 minutes | verifier_passed=True | retries=0 | correct=True

[tricky] Alice has 3 red apples and twice as many green apples as red. She gives away 2 green apples. How many does she have now?
  answer=7 apples | verifier_passed=True | retries=0 | correct=True

[tricky] A meeting needs 45 minutes. Free slots: 10:00-10:45, 11:00-11:40. Which slots can fit it?
  answer=10:00-10:45 | verifier_passed=True | retries=0 | correct=True

Logged questions: 8 | API calls made this run: 19


In [3]:
import json
r = json.load(open("run_logs.json"))
print(len(r), "questions logged")
for x in r:
    print(f"[{x['level']}] {x['question'][:50]}... -> {x['output']['answer']} | success={x['verifier_passed']} | retries={x['output']['metadata']['retries']}")

8 questions logged
[easy] If a train leaves at 14:30 and arrives at 18:05, h... -> 3 hours and 35 minutes | success=True | retries=0
[easy] Alice has 3 red apples and twice as many green app... -> 9 apples | success=True | retries=0
[easy] A meeting needs 60 minutes. Free slots: 09:00-09:3... -> 11:00-12:00 | success=True | retries=0
[easy] A pen costs 12 rupees. How much do 7 pens cost?... -> 84 rupees | success=True | retries=0
[easy] Ravi is 25. His sister is 7 years younger. What is... -> 43 | success=True | retries=0
[tricky] A train leaves at 23:45 and arrives at 02:10 the n... -> 2 hours and 25 minutes | success=True | retries=0
[tricky] Alice has 3 red apples and twice as many green app... -> 7 apples | success=True | retries=0
[tricky] A meeting needs 45 minutes. Free slots: 10:00-10:4... -> 10:00-10:45 | success=True | retries=0
